# Data Exploration — SC 2026

Explores the production Darshan dataset (131K logs from ALCF Polaris)
and the benchmark ground-truth dataset (623 logs from 6 benchmark suites).

**Expected runtime**: < 1 minute

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DIMS = ['access_granularity', 'metadata_intensity', 'parallelism_efficiency',
        'access_pattern', 'interface_choice', 'file_strategy',
        'throughput_utilization', 'healthy']

## 1. Production Dataset Overview

In [ ]:
prod_feat = pd.read_parquet('../data/processed/production/features.parquet')
prod_labels = pd.read_parquet('../data/processed/production/labels.parquet')

print(f'Production features: {prod_feat.shape}')
print(f'Production labels: {prod_labels.shape}')
print()
print('Heuristic label distribution:')
for dim in DIMS:
    n = int(prod_labels[dim].sum())
    print(f'  {dim:30s} {n:6d} ({100*n/len(prod_labels):5.1f}%)')

## 2. Benchmark Ground-Truth Overview

In [ ]:
bench_feat = pd.read_parquet('../data/processed/benchmark/features.parquet')
bench_labels = pd.read_parquet('../data/processed/benchmark/labels.parquet')

print(f'Benchmark features: {bench_feat.shape}')
print(f'Benchmark labels: {bench_labels.shape}')
print()
print('GT label distribution:')
for dim in DIMS:
    n = int(bench_labels[dim].sum())
    print(f'  {dim:30s} {n:4d} ({100*n/len(bench_labels):5.1f}%)')
print()
print('Per benchmark:')
for b in bench_labels['benchmark'].unique():
    mask = bench_labels['benchmark'] == b
    n = mask.sum()
    active = {d: int(bench_labels.loc[mask, d].sum()) for d in DIMS if bench_labels.loc[mask, d].sum() > 0}
    print(f'  {b}: {n} samples, labels: {active}')

## 3. Feature Distribution Comparison

In [ ]:
# Compare key features between production and benchmark
key_features = ['nprocs', 'runtime_seconds', 'POSIX_BYTES_WRITTEN', 
                'small_io_ratio', 'seq_write_ratio', 'metadata_time_ratio']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, key_features):
    if feat in prod_feat.columns and feat in bench_feat.columns:
        prod_vals = prod_feat[feat].dropna().values
        bench_vals = bench_feat[feat].dropna().values
        ax.hist(np.log1p(prod_vals), bins=50, alpha=0.6, label='Production', density=True)
        ax.hist(np.log1p(bench_vals), bins=30, alpha=0.6, label='Benchmark', density=True)
        ax.set_title(feat, fontsize=10)
        ax.legend(fontsize=8)
plt.suptitle('Feature Distribution: Production vs Benchmark (log1p scale)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Label Co-occurrence Analysis

In [ ]:
# Multi-label co-occurrence for production data
cooc = prod_labels[DIMS].T.dot(prod_labels[DIMS])
# Normalize by diagonal
diag = np.diag(cooc.values).astype(float)
diag[diag == 0] = 1
cooc_norm = cooc / diag[:, None]

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cooc_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[d[:12] for d in DIMS],
            yticklabels=[d[:12] for d in DIMS], ax=ax)
ax.set_title('Label Co-occurrence (P(col|row) in production data)', fontsize=12)
plt.tight_layout()
plt.show()